In [8]:
#Data Manipulation specific
import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe
from scipy.stats import norm
import scipy.stats as stats
from scipy import signal
from netCDF4 import Dataset
from osgeo import gdal

#Visualisation specific
import seaborn as sns 
import matplotlib.pyplot as plt
%matplotlib inline
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import mapping
import geopandas as gpd

#CRU TS4.09 data was downloaded (not available in this repo because of size limits) from: https://crudata.uea.ac.uk/cru/data/hrg/

#ERA5 data was downloaded (available in this repo) from: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=overview

#NOAA CPC: https://www.cpc.ncep.noaa.gov/data/teledoc/nao.shtml

#NOAA PSL: https://psl.noaa.gov/data/timeseries/month/DS/Nino34/


In [ ]:
z850 = xr.open_dataset('../../Article3_data/Z850.nc', engine='netcdf4')
precip = xr.open_dataset('../../Article3_data/CDR-1983-2025.nc', engine='netcdf4')
nao = xr.open_dataset('../../Article3_data/NAO_index.nc', engine='netcdf4')
nino34 = xr.open_dataset('../../Article3_data/NOAAPSL-HadISST-Nino34.nc', engine='netcdf4')
mo = xr.open_dataset('../../Article3_data/MO_index.nc', engine='netcdf4')
wemo = xr.open_dataset('../../Article3_data/WeMO_index.nc', engine='netcdf4')
morocco = "/Users/workad/Workspace/shapfiles/Morocco/Morocco.shp"

era5_slp = xr.open_dataset('../../A4-DroughtForecasting-Preprocessing/data/era5-slp.nc', engine='netcdf4')



SyntaxError: invalid syntax (2554706599.py, line 7)

**II- Compute WeMO using ERA5 mean sea level pressure**

In [ ]:
# ---------------------------------------------------------------------------
# WeMO, canonical construction: z(west) - z(east)
# Written to a NEW file; WeMO_index.nc (difference-then-standardise) is untouched.
# ---------------------------------------------------------------------------

# -------------------------------
# 1. Load ERA5 mean sea level pressure
# -------------------------------
ds = xr.open_dataset('../../Article3_data/wemo_era5_mslp.nc', engine='netcdf4')
ds = ds.drop_vars(['expver', 'number'], errors='ignore')

# -------------------------------
# 2. Poles (Martin-Vide & Lopez-Bustins 2006): San Fernando (Cadiz) and Padua
# -------------------------------
lat_west, lon_west = 36.25, -6.0
lat_east, lon_east = 45.5, 11.5

CLIM_START, CLIM_END = '1981-01-01', '2010-12-31'
VARNAME = 'WeMO_std'
LONG_NAME = 'Western Mediterranean Oscillation Index (difference of standardised poles)'
OUTFILE = '../../Article3_data/WeMO_index_zfirst.nc'

# -------------------------------
# 3. Longitude convention + pole-coverage guard
# -------------------------------
# ERA5 is served on either 0-360 or -180..180; convert if needed.
if float(ds.longitude.max()) > 180:
    lon_west = lon_west % 360
    lon_east = lon_east % 360
    print("longitude convention: 0-360 (poles converted)")
else:
    print(f"longitude convention: -180..180 "
          f"[{float(ds.longitude.min()):.2f} to {float(ds.longitude.max()):.2f}]")

def require_inside(lat, lon, label):
    """Refuse a pole that lies outside the file.

    .sel(method='nearest') snaps silently to the domain edge, which returns a
    plausible-looking series from the wrong place. Fail instead, and report how
    far the nearest grid point sits from the requested coordinate.
    """
    lat_min, lat_max = float(ds.latitude.min()), float(ds.latitude.max())
    lon_min, lon_max = float(ds.longitude.min()), float(ds.longitude.max())
    if not (lat_min <= lat <= lat_max and lon_min <= lon <= lon_max):
        raise ValueError(
            f"{label} pole ({lat}N, {lon}E) is OUTSIDE this file "
            f"(lat {lat_min} to {lat_max}, lon {lon_min} to {lon_max}). "
            f"method='nearest' would snap it to the edge and build the index from "
            f"the wrong location. Download an MSLP box that contains the pole.")
    got = ds.msl.sel(latitude=lat, longitude=lon, method="nearest")
    d_km = ((float(got.latitude) - lat) ** 2 + (float(got.longitude) - lon) ** 2) ** 0.5 * 111
    print(f"  {label:<6} requested ({lat}N, {lon}E) -> grid ({float(got.latitude)}N, "
          f"{float(got.longitude)}E), {d_km:.0f} km away")
    return got

msl_west = require_inside(lat_west, lon_west, "west")
msl_east = require_inside(lat_east, lon_east, "east")

# -------------------------------
# 4. Standardise EACH pole first, then difference
# -------------------------------
# Martin-Vide & Lopez-Bustins (2006) define WeMOi as the difference of the
# standardised pressures. Differencing raw pressure instead lets whichever pole
# has the larger variance dominate the index - see index_construction_check.md.
df = pd.DataFrame({"west": msl_west.values, "east": msl_east.values},
                  index=pd.to_datetime(ds["valid_time"].values))

clim = df.loc[CLIM_START:CLIM_END]

def standardise_by_month(col):
    mean = clim[col].groupby(clim.index.month).mean()
    std = clim[col].groupby(clim.index.month).std()
    return (df[col] - df.index.month.map(mean)) / df.index.month.map(std)

df["z_west"] = standardise_by_month("west")
df["z_east"] = standardise_by_month("east")
df[VARNAME] = df["z_west"] - df["z_east"]
print(f"  {VARNAME}: n = {df[VARNAME].notna().sum()}, sd = {df[VARNAME].std():.3f}")

# -------------------------------
# 5. Save
# -------------------------------
out = xr.Dataset({VARNAME: (["time"], df[VARNAME].values)},
                 coords={"time": df.index.values})
out[VARNAME].attrs["long_name"] = LONG_NAME
out[VARNAME].attrs["units"] = "unitless"
out[VARNAME].attrs["construction"] = ("difference of the per-calendar-month standardised "
                                      "pole pressures (z_west - z_east)")
out.attrs["climatology_period"] = f"{CLIM_START} to {CLIM_END}"
out.attrs["west_pole"] = f"{lat_west}N {lon_west}E"
out.attrs["east_pole"] = f"{lat_east}N {lon_east}E"
out.to_netcdf(OUTFILE)
print(f"Saved: {OUTFILE}")


**III- Compute MO using ERA5 mean sea level pressure**

In [ ]:
# ---------------------------------------------------------------------------
# MO index: both pole pairs (MOI1 / MOI2) and both construction orders
# Written to a NEW file; MO_index.nc (difference-then-standardise) is untouched.
#
# NOTE ON VARIANTS (CRU/UEA):
#   MOI2 = Gibraltar N. Frontier (36.1N, 5.3W) - Lod Airport (32.0N, 34.5E)
#   MOI1 = Algiers (36.4N, 3.1E) - Cairo (30.1N, 31.4E)
# mo_era5_mslp.nc spans lat 28-42N, lon 10W-35E (1981-2025) and contains BOTH pole
# pairs. The guard in step 3 still refuses any pole outside the file rather than
# letting .sel(method='nearest') snap it silently to the edge.
# ---------------------------------------------------------------------------

# -------------------------------
# 1. Load ERA5 mean sea level pressure
# -------------------------------
ds = xr.open_dataset('../../Article3_data/mo_era5_mslp.nc', engine='netcdf4')
ds = ds.drop_vars(['expver', 'number'], errors='ignore')
#ds = ds.sel(valid_time=slice('1950-01-01', '2024-12-31'))

# -------------------------------
# 2. Pole coordinates
# -------------------------------
VARIANT = 'MOI1'          # 'MOI2' = Gibraltar-Lod | 'MOI1' = Algiers-Cairo (needs a wider file)
POLES = {
    'MOI2': {'west': (36.0, -5.0), 'east': (32.0, 34.5)},
    'MOI1': {'west': (36.5,  3.0), 'east': (30.0, 31.5)},
}
lat_west, lon_west = POLES[VARIANT]['west']
lat_east, lon_east = POLES[VARIANT]['east']

# Construction order (see index_construction_check.md):
#   'z_first'    = z(west) - z(east)           [Martin-Vide's WeMOi convention]
#   'diff_first' = standardise(west - east)    [CRU/Palutikof's published MOI form]
CONSTRUCTION = 'z_first'
_TAG = {'z_first': 'zfirst', 'diff_first': 'difffirst'}[CONSTRUCTION]

CLIM_START, CLIM_END = '1981-01-01', '2010-12-31'
VARNAME = 'MO_std'
LONG_NAME = f'Mediterranean Oscillation Index {VARIANT} ({CONSTRUCTION})'
OUTFILE = f'../../Article3_data/MO_index_{VARIANT}_{_TAG}.nc'

# -------------------------------
# 3. Longitude convention + pole-coverage guard
# -------------------------------
# ERA5 is served on either 0-360 or -180..180; convert if needed.
if float(ds.longitude.max()) > 180:
    lon_west = lon_west % 360
    lon_east = lon_east % 360
    print("longitude convention: 0-360 (poles converted)")
else:
    print(f"longitude convention: -180..180 "
          f"[{float(ds.longitude.min()):.2f} to {float(ds.longitude.max()):.2f}]")

def require_inside(lat, lon, label):
    """Refuse a pole that lies outside the file.

    .sel(method='nearest') snaps silently to the domain edge, which returns a
    plausible-looking series from the wrong place. Fail instead, and report how
    far the nearest grid point sits from the requested coordinate.
    """
    lat_min, lat_max = float(ds.latitude.min()), float(ds.latitude.max())
    lon_min, lon_max = float(ds.longitude.min()), float(ds.longitude.max())
    if not (lat_min <= lat <= lat_max and lon_min <= lon <= lon_max):
        raise ValueError(
            f"{label} pole ({lat}N, {lon}E) is OUTSIDE this file "
            f"(lat {lat_min} to {lat_max}, lon {lon_min} to {lon_max}). "
            f"method='nearest' would snap it to the edge and build the index from "
            f"the wrong location. Download an MSLP box that contains the pole.")
    got = ds.msl.sel(latitude=lat, longitude=lon, method="nearest")
    d_km = ((float(got.latitude) - lat) ** 2 + (float(got.longitude) - lon) ** 2) ** 0.5 * 111
    print(f"  {label:<6} requested ({lat}N, {lon}E) -> grid ({float(got.latitude)}N, "
          f"{float(got.longitude)}E), {d_km:.0f} km away")
    return got

msl_west = require_inside(lat_west, lon_west, "west")
msl_east = require_inside(lat_east, lon_east, "east")

# -------------------------------
# 4. Standardise EACH pole first, then difference
# -------------------------------
# Martin-Vide & Lopez-Bustins (2006) define WeMOi as the difference of the
# standardised pressures. Differencing raw pressure instead lets whichever pole
# has the larger variance dominate the index - see index_construction_check.md.
df = pd.DataFrame({"west": msl_west.values, "east": msl_east.values},
                  index=pd.to_datetime(ds["valid_time"].values))

clim = df.loc[CLIM_START:CLIM_END]

def standardise_by_month(col):
    mean = clim[col].groupby(clim.index.month).mean()
    std = clim[col].groupby(clim.index.month).std()
    return (df[col] - df.index.month.map(mean)) / df.index.month.map(std)

if CONSTRUCTION == "z_first":
    df["z_west"] = standardise_by_month("west")
    df["z_east"] = standardise_by_month("east")
    df[VARNAME] = df["z_west"] - df["z_east"]
else:                                    # 'diff_first': the published MOI form
    df["raw"] = df["west"] - df["east"]
    clim = df.loc[CLIM_START:CLIM_END]   # recompute on the difference column
    df[VARNAME] = standardise_by_month("raw")
print(f"  {VARNAME}: n = {df[VARNAME].notna().sum()}, sd = {df[VARNAME].std():.3f}")

# -------------------------------
# 5. Save
# -------------------------------
out = xr.Dataset({VARNAME: (["time"], df[VARNAME].values)},
                 coords={"time": df.index.values})
out[VARNAME].attrs["long_name"] = LONG_NAME
out[VARNAME].attrs["units"] = "unitless"
out[VARNAME].attrs["construction"] = CONSTRUCTION
out.attrs["climatology_period"] = f"{CLIM_START} to {CLIM_END}"
out.attrs["west_pole"] = f"{lat_west}N {lon_west}E"
out.attrs["east_pole"] = f"{lat_east}N {lon_east}E"
out.to_netcdf(OUTFILE)
print(f"Saved: {OUTFILE}")


**III- Validation of ERA5 derived MO and WeMO against published indices:**

In [ ]:
era5_mo = xr.open_dataset('../../Article3_data/MO_index.nc', engine='netcdf4')
era5_wemo = xr.open_dataset('../../Article3_data/WeMO_index.nc', engine='netcdf4')


<xarray.Dataset> Size: 11kB
Dimensions:  (time: 900)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2024-12-01
Data variables:
    MO_std   (time) float32 4kB ...
Attributes:
    climatology_period:  1981-01-01 to 2010-12-31

In [12]:
# ---------------------------------------------------------------------------
# Validation of the ERA5-derived indices against the published series
# ---------------------------------------------------------------------------

# -------------------------------
# 1. Published MO (daily, fixed width: "YYYY.MM.DD.  value") -> monthly mean
# -------------------------------
mo_raw = pd.read_fwf('../../Article3_data/moi2.output.dat',
                     colspecs=[(0, 4), (5, 7), (8, 10), (11, 18)],
                     names=['year', 'month', 'day', 'MO'])

mo_daily = pd.Series(mo_raw['MO'].values,
                     index=pd.to_datetime(mo_raw[['year', 'month', 'day']]))

n_days = mo_daily.resample('MS').count()
pub_mo = mo_daily.resample('MS').mean().where(n_days >= 28)   # only complete months

# -------------------------------
# 2. Published WeMO (monthly table: year x Jan..Dec, comma decimal, -999 = missing)
# -------------------------------
wemo_tab = pd.read_csv('../../Article3_data/Web_WeMOi-2020.txt',
                       sep='\t', header=None, skiprows=9, usecols=range(13),
                       names=['year'] + list(range(1, 13)), dtype=str)
wemo_tab = wemo_tab.apply(lambda c: pd.to_numeric(c.str.replace(',', '.', regex=False),
                                                  errors='coerce'))
wemo_tab = wemo_tab.dropna(subset=['year']).set_index('year')

wemo_long = wemo_tab.stack().replace(-999, np.nan).dropna()
pub_wemo = pd.Series(
    wemo_long.values,
    index=pd.to_datetime(dict(year=wemo_long.index.get_level_values(0).astype(int),
                              month=wemo_long.index.get_level_values(1).astype(int),
                              day=1))).sort_index()

print(f"Published MO   : {pub_mo.index[0]:%Y-%m} to {pub_mo.index[-1]:%Y-%m}  "
      f"({pub_mo.notna().sum()} monthly values from {len(mo_daily)} daily values)")
print(f"Published WeMO : {pub_wemo.index[0]:%Y-%m} to {pub_wemo.index[-1]:%Y-%m}  "
      f"({pub_wemo.notna().sum()} monthly values)")
print(f"ERA5 MO        : {pd.Timestamp(era5_mo.time.values[0]):%Y-%m} to "
      f"{pd.Timestamp(era5_mo.time.values[-1]):%Y-%m}  ({era5_mo.sizes['time']} monthly values)")
print(f"ERA5 WeMO      : {pd.Timestamp(era5_wemo.time.values[0]):%Y-%m} to "
      f"{pd.Timestamp(era5_wemo.time.values[-1]):%Y-%m}  ({era5_wemo.sizes['time']} monthly values)")

# -------------------------------
# 3. Overlap period + correlation
# -------------------------------
def standardize_monthly(s, clim_start='1981', clim_end='2010'):
    """Same monthly (1981-2010) standardization applied to the ERA5 indices."""
    clim = s.loc[clim_start:clim_end]
    mean = clim.groupby(clim.index.month).mean()
    std  = clim.groupby(clim.index.month).std()
    return (s - s.index.month.map(mean)) / s.index.month.map(std)

def validate(name, era5_da, published):
    era5 = era5_da.to_series()
    era5.index = era5.index.to_period('M').to_timestamp()
    published = published.copy()
    published.index = published.index.to_period('M').to_timestamp()

    # overlap = months present (and non-missing) in both series
    both = pd.concat({'era5': era5, 'published': published}, axis=1,
                     join='inner').dropna()

    r,   p_r   = stats.pearsonr(both['era5'], both['published'])
    rho, p_rho = stats.spearmanr(both['era5'], both['published'])

    # like-for-like: published series put on the same monthly standardization
    pub_std = standardize_monthly(both['published'])
    r_std, p_std = stats.pearsonr(both['era5'], pub_std)

    print(f"\n{name}  overlap {both.index[0]:%Y-%m} to {both.index[-1]:%Y-%m}  (n = {len(both)} months)")
    print(f"   Pearson  r = {r:.3f}  (p = {p_r:.2e})")
    print(f"   Spearman rho = {rho:.3f}  (p = {p_rho:.2e})")
    print(f"   Pearson  r = {r_std:.3f}  after applying the same 1981-2010 "
          f"monthly standardization to the published series")
    return both

both_mo   = validate('MO   (ERA5 vs published MOI2, Gibraltar-Lod)', era5_mo['MO_std'],     pub_mo)
both_wemo = validate('WeMO (ERA5 vs Martin-Vide & Lopez-Bustins)', era5_wemo['WeMO_std'], pub_wemo)

# -------------------------------
# 4. Seasonal correlation (both series standardized over 1981-2010)
# -------------------------------
seasons = {'SON': [9, 10, 11], 'DJF': [12, 1, 2], 'MAM': [3, 4, 5]}

def seasonal_corr(name, both):
    era5 = both['era5']
    pub  = standardize_monthly(both['published'])
    print(f"\n{name}  (overlap {both.index[0]:%Y-%m} to {both.index[-1]:%Y-%m})")
    for season, months in seasons.items():
        m = both.index.month.isin(months)
        r_mon, p_mon = stats.pearsonr(era5[m], pub[m])

        # seasonal means: December is assigned to the following year's DJF
        key = pd.Series(both.index.year + (both.index.month == 12), index=both.index)[m]
        full = key.groupby(key).transform('size') == 3          # complete seasons only
        era5_seas = era5[m][full].groupby(key[full]).mean()
        pub_seas  = pub[m][full].groupby(key[full]).mean()
        r_seas, p_seas = stats.pearsonr(era5_seas, pub_seas)

        print(f"   {season}:  monthly r = {r_mon:.3f} (n = {m.sum()}, p = {p_mon:.1e})"
              f"   season-mean r = {r_seas:.3f} (n = {len(era5_seas)}, p = {p_seas:.1e})")

seasonal_corr('MO   (ERA5 vs published MOI2, Gibraltar-Lod)', both_mo)
seasonal_corr('WeMO (ERA5 vs Martin-Vide & Lopez-Bustins)', both_wemo)

Published MO   : 1948-01 to 2022-12  (900 monthly values from 27394 daily values)
Published WeMO : 1821-01 to 2020-12  (2372 monthly values)
ERA5 MO        : 1950-01 to 2024-12  (900 monthly values)
ERA5 WeMO      : 1981-01 to 2025-12  (540 monthly values)

MO   (ERA5 vs Conte et al. MOI2)  overlap 1950-01 to 2022-12  (n = 876 months)
   Pearson  r = 0.668  (p = 2.41e-114)
   Spearman rho = 0.637  (p = 1.01e-100)
   Pearson  r = 0.910  after applying the same 1981-2010 monthly standardization to the published series

WeMO (ERA5 vs Martin-Vide & Lopez-Bustins)  overlap 1981-01 to 2020-12  (n = 480 months)
   Pearson  r = 0.849  (p = 1.18e-134)
   Spearman rho = 0.846  (p = 7.49e-133)
   Pearson  r = 0.856  after applying the same 1981-2010 monthly standardization to the published series

MO   (ERA5 vs Conte et al. MOI2)  (overlap 1950-01 to 2022-12)
   SON:  monthly r = 0.934 (n = 219, p = 1.2e-98)   season-mean r = 0.906 (n = 73, p = 2.8e-28)
   DJF:  monthly r = 0.986 (n = 219, p = 3.

**IV- Build the circulation predictor Z850**

In [ ]:
z850 = xr.open_dataset('../../Article3_data/Z850.nc', engine='netcdf4')


<xarray.Dataset> Size: 10MB
Dimensions:         (valid_time: 540, pressure_level: 1, latitude: 69,
                     longitude: 69)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 4kB 1980-01-01 ... 2024-12-01
    expver          (valid_time) <U4 9kB ...
  * pressure_level  (pressure_level) float64 8B 850.0
  * latitude        (latitude) float64 552B 37.0 36.75 36.5 ... 20.5 20.25 20.0
  * longitude       (longitude) float64 552B -18.0 -17.75 -17.5 ... -1.25 -1.0
    number          int64 8B ...
Data variables:
    z               (valid_time, pressure_level, latitude, longitude) float32 10MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-12T15:30 GRIB to CDM+CF via cfgrib-0.9.1...

In [29]:
nino34 = xr.open_dataset('data/NOAAPSL-HadISST-Nino34.nc')
nino34 = nino34.sel(time=slice('1950-01-01', '2024-12-31'))
nino34

<xarray.Dataset> Size: 11kB
Dimensions:  (time: 900)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2024-12-01
Data variables:
    value    (time) float32 4kB ...

In [31]:
# -------------------------------
# 6. Merge all three Climate indices into one dataset
# -------------------------------

nino34 = xr.open_dataset('data/NOAAPSL-HadISST-Nino34.nc')
nino34 = nino34.sel(time=slice('1950-01-01', '2024-12-31'))

nao = xr.open_dataset('data/NAO_index.nc')
nao = nao["NAO"]

mo = xr.open_dataset('data/MO_index.nc')
mo = mo["MO_std"]

# Merge all datasets
ds_out = xr.merge([nino34, nao, mo], join="inner")

# Rename Nino34 variable if needed (it might be called 'value')
if "value" in ds_out.data_vars:
    ds_out = ds_out.rename({"value": "Nino34"})

# Add metadata
ds_out.attrs["title"] = "Merged Teleconnection Indices"
ds_out.attrs["climatology_period"] = "1981-2010"

# -------------------------------
# 7. Save dataset
# -------------------------------
output_file = "output/climate_indices.nc"
ds_out.to_netcdf(output_file)

print(ds_out)
print(f"Saved: {output_file}")

<xarray.Dataset> Size: 22kB
Dimensions:  (time: 900)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2024-12-01
Data variables:
    Nino34   (time) float32 4kB ...
    NAO      (time) float64 7kB ...
    MO_std   (time) float32 4kB ...
Attributes:
    title:               Merged Teleconnection Indices
    climatology_period:  1981-2010
Saved: output/climate_indices.nc


**IV- Aligning CRU and ERA5 data using xESMF**

**V- ERA5 preprocessing**

**VI- Dataframes preparation for SPEI3 calculation using R**

**VII- Preparing and merging CRU data in one .nc file (Standardization omitted at this stage)**